# 02 Baseline Ranking Vendor

Notebook ini menutup tahap **Data Preparation** dan menetapkan **baseline** yang nanti harus dikalahkan model LightGBM LambdaRank.

- Dataset: `ml/data/processed/dataset.csv`, dibangun dengan `python -m procurewise_ml.build_dataset` dari data seed 15037
- Fitur: `procurewise_ml/features.py`, fungsi murni yang nanti juga dipanggil backend
- Baseline: `procurewise_ml/baselines.py`, bobot di `procurewise_ml/config.py`
- Evaluasi: `procurewise_ml/evaluation.py`, dipakai ulang untuk semua model

Hasil utama ada di split **test** (1 Juli s.d. 30 September 2026).

In [1]:
import sys
from pathlib import Path

import pandas as pd


def find_ml_dir() -> Path:
    # notebook bisa dijalankan dari ml/notebooks, ml, atau root repo
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / "ml"):
            if (candidate / "procurewise_ml" / "__init__.py").exists():
                return candidate
    raise FileNotFoundError("folder ml tidak ditemukan")


sys.path.insert(0, str(find_ml_dir()))

from procurewise_ml import config
from procurewise_ml import evaluation as ev
from procurewise_ml.baselines import BASELINES
from procurewise_ml.data import load_dataset

dataset, spec = load_dataset()
assert spec["source"]["seed"] == 15037, "notebook ini disusun untuk data seed 15037"
print(f"seed {spec['source']['seed']}, {spec['rows']:,} pasangan request dan vendor")
pd.DataFrame({"request": spec["requests_per_split"], "grup ranking": spec["ranking_groups_per_split"]})

seed 15037, 9,754 pasangan request dan vendor


,request,grup ranking
train,1475,1360
val,397,364
test,230,218


## 1. Daftar Fitur Final

Satu baris per pasangan (request, vendor kandidat). Fitur histori hanya memakai transaksi yang hasilnya sudah diketahui sebelum `request_date`. Vendor tanpa histori memakai rata-rata kategorinya sampai `request_date` dan diberi `has_history = 0`.

In [2]:
descriptions = {
    "unit_price": "harga per satuan pada snapshot penawaran saat request",
    "price_rel_median": "unit_price dibagi median harga semua penawaran material itu di request yang sama",
    "total_cost": "unit_price x qty",
    "quoted_lead_time_days": "janji lead time vendor",
    "moq_ratio": "moq dibagi qty",
    "on_time_rate": "proporsi transaksi selesai yang tiba sebelum atau pada promised_date",
    "avg_delay_days": "rata-rata hari telat terhadap promised_date (telat negatif dihitung 0)",
    "defect_rate": "total qty_defect dibagi total qty transaksi selesai",
    "avg_rating": "rata-rata quality_rating transaksi selesai",
    "n_transactions": "jumlah transaksi selesai yang sudah diterima sebelum request_date",
    "days_since_last_order": "hari sejak order terakhir ke vendor itu (maks. 365)",
    "has_history": "1 kalau vendor punya transaksi selesai sebelum request_date",
    "slack_days": "(needed_by - request_date) - quoted_lead_time_days",
    "qty_capacity_ratio": "qty dibagi weekly_capacity",
    "priority_code": "priority request: low 0, normal 1, high 2",
    "is_peak_season": "1 kalau request_date jatuh di musim ramai (kalender bisnis)",
}
groups = {
    **{f: "penawaran" for f in config.OFFER_FEATURES},
    **{f: "histori" for f in config.HISTORY_FEATURES},
    **{f: "konteks" for f in config.CONTEXT_FEATURES},
}
eligible = dataset[dataset["is_eligible"]]
feature_table = pd.DataFrame({
    "kelompok": pd.Series(groups),
    "keterangan": pd.Series(descriptions),
    "median": eligible[list(config.FEATURE_COLUMNS)].median(),
    "korelasi_dengan_grade": eligible[list(config.FEATURE_COLUMNS)].corrwith(
        eligible["grade"].astype(float), method="spearman"),
}).loc[list(config.FEATURE_COLUMNS)]
feature_table.round(3)

,kelompok,keterangan,median,korelasi_dengan_grade
unit_price,penawaran,harga per satuan pada snapshot penawaran saat ...,7700.000,-0.049
price_rel_median,penawaran,unit_price dibagi median harga semua penawaran...,1.000,-0.232
total_cost,penawaran,unit_price x qty,1377000.000,-0.120
quoted_lead_time_days,penawaran,janji lead time vendor,9.000,-0.134
moq_ratio,penawaran,moq dibagi qty,0.227,0.138
on_time_rate,histori,proporsi transaksi selesai yang tiba sebelum a...,0.778,0.248
avg_delay_days,histori,rata-rata hari telat terhadap promised_date (t...,0.866,-0.201
defect_rate,histori,total qty_defect dibagi total qty transaksi se...,0.023,-0.363
avg_rating,histori,rata-rata quality_rating transaksi selesai,3.000,0.434
n_transactions,histori,jumlah transaksi selesai yang sudah diterima s...,20.000,0.092


**Temuan:** Fitur dengan hubungan terkuat ke grade adalah `avg_rating` (+0,43), `slack_days` (+0,37), dan `defect_rate` (-0,36), dan tidak ada fitur yang korelasinya mendekati sempurna. Fitur histori dan konteks sama-sama penting, sedangkan harga mentah (`unit_price`) hampir tidak berkorelasi karena skalanya berbeda antar material sehingga yang berguna adalah harga relatif.

## 2. Request yang Dikeluarkan dari Metrik Ranking

Request dengan kurang dari 2 kandidat lolos filter keras tidak bisa di-ranking, jadi dikeluarkan dari NDCG, Hit@1, dan MRR.

In [3]:
coverage_table = ev.coverage(dataset)
coverage_view = coverage_table.astype({"n_request": int, "dikeluarkan": int, "tanpa_kandidat_lolos": int})
coverage_view["persen_dikeluarkan"] = coverage_table["persen_dikeluarkan"].map("{:.1%}".format)
coverage_view

,n_request,dikeluarkan,tanpa_kandidat_lolos,persen_dikeluarkan
material_category,,,,
bahan,880,34,5,3.9%
jasa,634,16,1,2.5%
packaging,588,110,46,18.7%
total,2102,160,52,7.6%


**Temuan:** Sebanyak 160 request (7,6%) dikeluarkan dari metrik ranking, dan 52 di antaranya sama sekali tidak punya kandidat yang lolos filter. Packaging paling terdampak (18,7%) karena banyak kandidatnya gugur di filter budget dan MOQ, sedangkan bahan dan jasa di bawah 4%.

## 3. Baseline dan Bobotnya

Ketiga baseline memakai lima kriteria dari fitur yang terlihat. Skor dihitung per request dari kandidat yang lolos filter.

- **weighted_sum**: tiap kriteria dinormalisasi min-max di dalam request (kriteria cost dibalik), lalu dijumlah dengan bobot tetap
- **topsis**: normalisasi vektor, lalu jarak ke solusi ideal dan anti-ideal dengan bobot tetap
- **topsis_priority**: sama dengan topsis, tapi bobot dipilih dari priority request

Sebagai pembanding juga ditampilkan **acak** (urutan acak, seed 0), **pilihan_historis** (vendor yang dulu dipilih Firaza), dan **oracle** (urutan dari label, batas atas).

In [4]:
weights = pd.DataFrame({
    "arah": dict(config.BASELINE_CRITERIA),
    "bobot_tetap": config.BASELINE_FIXED_WEIGHTS,
    **{f"priority_{p}": w for p, w in config.BASELINE_PRIORITY_WEIGHTS.items()},
})
weights

,arah,bobot_tetap,priority_high,priority_normal,priority_low
total_cost,cost,0.30,0.15,0.30,0.45
quoted_lead_time_days,cost,0.15,0.25,0.15,0.10
on_time_rate,benefit,0.25,0.35,0.25,0.15
defect_rate,cost,0.20,0.15,0.20,0.20
avg_rating,benefit,0.10,0.10,0.10,0.10


In [5]:
scores = {name: fn(dataset) for name, fn in BASELINES.items()}
scores |= ev.reference_scores(dataset)


def view(table: pd.DataFrame) -> pd.DataFrame:
    out = table.copy()
    out["n_grup"] = out["n_grup"].astype(int)
    for column in ("ndcg@3", "hit@1", "mrr", "biaya_relatif", "rata_grade"):
        if column in out:
            out[column] = out[column].astype(float).round(3)
    for column in ("on_time", "defect_rate", "top1_grade_0"):
        if column in out:
            out[column] = out[column].astype(float).map("{:.1%}".format)
    if "total_biaya" in out:
        out["total_biaya"] = (out["total_biaya"].astype(float) / 1e6).map("Rp{:,.1f} jt".format)
    return out

## 4. Hasil di Split Test

Kolom metrik bisnis dihitung untuk vendor peringkat 1 di setiap grup, memakai hasil simulasi dari label: `on_time` adalah peluang tiba sebelum `needed_by`, `defect_rate` adalah perkiraan proporsi cacat, `total_biaya` adalah jumlah biaya efektif (termasuk pembelian ulang barang cacat), dan `biaya_relatif` adalah biaya vendor peringkat 1 dibagi biaya kandidat termurah di grup yang sama.

In [6]:
test_table = ev.compare(dataset, scores, "test")
view(test_table)

,n_grup,ndcg@3,hit@1,mrr,on_time,defect_rate,total_biaya,biaya_relatif,rata_grade,top1_grade_0
weighted_sum,218,0.937,0.853,0.918,90.9%,1.5%,Rp356.7 jt,1.042,2.500,2.8%
topsis,218,0.912,0.798,0.885,89.8%,1.2%,Rp379.0 jt,1.116,2.440,3.2%
topsis_priority,218,0.933,0.826,0.908,90.7%,1.3%,Rp379.0 jt,1.099,2.482,1.8%
acak,218,0.778,0.459,0.674,85.2%,2.3%,Rp387.1 jt,1.158,1.904,7.8%
pilihan_historis,218,0.765,0.495,0.661,87.3%,3.1%,Rp352.4 jt,1.039,1.991,6.9%
oracle,218,1.000,1.000,1.000,94.1%,1.3%,Rp365.2 jt,1.074,2.670,0.0%


**Temuan:** Ketiga baseline jauh di atas urutan acak dan pilihan historis Firaza, dengan weighted sum paling baik (NDCG@3 0,937 dan Hit@1 0,853). Vendor peringkat 1 versi weighted sum tepat waktu 90,9% dengan defect 1,5%, lebih baik dari pilihan historis (87,3% dan 3,1%) dengan biaya relatif yang hampir sama (1,04 kali kandidat termurah).

### Grup yang Grade-nya Bervariasi

Grup yang semua kandidatnya punya grade sama selalu mendapat nilai sempurna untuk model apa pun. Tabel ini hanya memakai grup dengan minimal dua grade berbeda supaya perbedaan antar baseline lebih terlihat.

In [7]:
informative = ev.compare(dataset, scores, "test", informative_only=True)
view(informative[["n_grup", "ndcg@3", "hit@1", "mrr"]])

,n_grup,ndcg@3,hit@1,mrr
weighted_sum,194,0.930,0.835,0.908
topsis,194,0.901,0.773,0.871
topsis_priority,194,0.925,0.804,0.897
acak,194,0.750,0.392,0.634
pilihan_historis,194,0.736,0.433,0.619
oracle,194,1.000,1.000,1.000


**Temuan:** Di 194 grup yang grade-nya bervariasi, urutannya tetap sama dan weighted sum masih unggul (Hit@1 0,835 berbanding 0,392 untuk acak). Tabel ini lebih tepat dipakai untuk membandingkan model nanti karena grup yang grade-nya sama semua tidak membedakan model apa pun.

## 5. Stabilitas Antar Split

Baseline tidak dilatih, jadi metriknya di train dan val bisa dipakai untuk melihat apakah test kebetulan mudah atau sulit.

In [8]:
rows = []
for name in BASELINES:
    table = ev.ranking_metrics(dataset, scores[name])
    for split, row in table.iterrows():
        rows.append({"baseline": name, "split": split, **row.to_dict()})
per_split = pd.DataFrame(rows).pivot(index="baseline", columns="split", values=["ndcg@3", "hit@1"])
per_split = per_split.reindex(columns=pd.MultiIndex.from_product([["ndcg@3", "hit@1"], config.SPLITS]))
per_split.round(3)

ndcg@3                hit@1              
                 train    val   test  train    val   test
baseline                                                 
topsis           0.898  0.914  0.912  0.774  0.835  0.798
topsis_priority  0.906  0.917  0.933  0.800  0.821  0.826
weighted_sum     0.911  0.916  0.937  0.816  0.846  0.853

**Temuan:** Metrik ketiga baseline stabil di train, val, dan test (NDCG@3 0,90 sampai 0,94) dan urutan antar baseline tidak berubah. Test sedikit lebih tinggi dari train, kemungkinan karena periode test hampir tanpa musim ramai sehingga janji lead time lebih bisa dipercaya.

## Ringkasan

1. **Target yang harus dikalahkan LightGBM** di split test adalah weighted sum: NDCG@3 0,937, Hit@1 0,853, dan MRR 0,918 (di grup informatif 0,930, 0,835, dan 0,908).
2. **Ruang perbaikan sempit tapi ada.** Oracle menunjukkan vendor peringkat 1 bisa tepat waktu 94,1% dengan rata-rata grade 2,67, sedangkan weighted sum baru 90,9% dan 2,50.
3. **Bobot dari priority membantu TOPSIS** (NDCG@3 naik dari 0,912 ke 0,933), tanda bahwa konteks request memang penting untuk ranking.
4. **Pilihan historis Firaza paling murah tapi paling sering cacat**, sehingga histori transaksi berisi bias seleksi yang perlu diingat saat melatih model.
5. **Hasil packaging perlu dibaca hati-hati** karena hampir seperlima request-nya tidak masuk metrik ranking.